In [7]:
import pandas as pd
import json
import csv
import os
import re

def clean_text(text):
    """Clean text by handling encoding issues and special characters"""
    if not isinstance(text, str):
        return str(text)
    
    # Handle the specific encoding issue you're seeing
    # This often happens when UTF-8 text is read as latin-1 or cp1252
    try:
        # Try to detect if this is a UTF-8 string that was incorrectly decoded
        text = re.sub(r'�+', "'", text)
        
        # Replace other common encoding issues
        replacements = {
            ''': "'",
            ''': "'", 
            '"': '"',
            '"': '"',
            '–': '-',
            '—': '-',
            '…': '...',
            '•': '*',
            'â€™': "'",
            'â€œ': '"',
            'â€': '"',
        }
        
        for old, new in replacements.items():
            text = text.replace(old, new)
            
    except Exception as e:
        print(f"Error cleaning text: {e}")
    
    # Clean up multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def load_movie_database(movie_db_file):
    """Load movie database and create a mapping from video_id to title"""
    try:
        # Try reading with UTF-8 first, then other encodings
        df = None
        successful_encoding = None
        
        encodings_to_try = [
            'utf-8',
            'utf-8-sig',  # UTF-8 with BOM
            'latin-1', 
            'cp1252', 
            'iso-8859-1'
        ]
        
        for encoding in encodings_to_try:
            try:
                df = pd.read_csv(movie_db_file, sep='\t', encoding=encoding, on_bad_lines='skip')
                successful_encoding = encoding
                print(f"Successfully loaded movie database with {encoding} encoding")
                break
            except (UnicodeDecodeError, pd.errors.ParserError) as e:
                print(f"Failed to load with {encoding}: {e}")
                continue
        
        if df is None:
            print("Could not load movie database with any encoding")
            return {}
        
        # Create a mapping from video_id to title
        video_id_to_title = {}
        for _, row in df.iterrows():
            video_id = str(row.get('video_id', '')).strip()
            title = str(row.get('title', '')).strip()
            
            if video_id and title and title != 'nan':
                # Clean up the title but preserve it
                cleaned_title = clean_text(title)
                video_id_to_title[video_id] = cleaned_title
                print(f"Mapped {video_id} -> {cleaned_title}")
        
        print(f"Loaded {len(video_id_to_title)} movies from database")
        return video_id_to_title
    except Exception as e:
        print(f"Error loading movie database: {e}")
        return {}

def extract_conversations(csv_file, movie_db_mapping):
    # Read the CSV file with proper encoding handling
    df = None
    successful_encoding = None
    
    encodings_to_try = [
        'utf-8',
        'utf-8-sig',  # UTF-8 with BOM
        'latin-1', 
        'cp1252', 
        'iso-8859-1'
    ]
    
    for encoding in encodings_to_try:
        try:
            df = pd.read_csv(csv_file, sep='\t', encoding=encoding, on_bad_lines='skip')
            successful_encoding = encoding
            print(f"Successfully loaded {csv_file} with {encoding} encoding")
            break
        except (UnicodeDecodeError, pd.errors.ParserError) as e:
            print(f"Failed to load {csv_file} with {encoding}: {e}")
            continue
    
    if df is None:
        print(f"Could not load {csv_file} with any encoding")
        return []
    
    # Dictionary to store all conversations
    all_conversations = []
    
    # Group by dialog_id to process each conversation
    for dialog_id in df['dialog_id'].unique():
        dialog_df = df[df['dialog_id'] == dialog_id].sort_values('utt_id')
        movie_id = str(df[df['dialog_id'] == dialog_id]['movie_id'].iloc[0]).strip()
        
        # Convert video_id to movie title
        movie_title = movie_db_mapping.get(movie_id, movie_id)  # Fallback to original if not found
        print(f"Dialog {dialog_id}: {movie_id} -> '{movie_title}'")
        
        conversation = []
        current_speaker = None
        current_content = []
        
        for _, row in dialog_df.iterrows():
            speaker = row['speaker']
            text = clean_text(str(row['text']))  # Clean the text content
            
            # If same speaker continues, accumulate text
            if speaker == current_speaker:
                current_content.append(text)
            else:
                # If we have accumulated content, add it to conversation
                if current_content and current_speaker is not None:
                    prev_role = 'user' if current_speaker == 'SEEKER' else 'assistant'
                    conversation.append({
                        'role': prev_role,
                        'content': ' '.join(current_content)
                    })
                
                # Start new speaker's content
                current_speaker = speaker
                current_content = [text]
        
        # Don't forget the last accumulated content
        if current_content and current_speaker is not None:
            role = 'user' if current_speaker == 'SEEKER' else 'assistant'
            conversation.append({
                'role': role,
                'content': ' '.join(current_content)
            })

        # Add conversation to the list if it has content
        if conversation:
            all_conversations.append({
                'dialog_id': dialog_id,
                'ground_truth': movie_title,  # Now using movie title instead of video_id
                'conversation': conversation
            })
    
    return all_conversations

def save_conversations_to_csv(conversations, output_file):
    """Save conversations directly to CSV format with proper encoding"""
    csv_data = []
    for entry in conversations:
        dialog_id = entry.get('dialog_id', '')
        conversation = entry.get('conversation', [])
        ground_truth = clean_text(entry.get('ground_truth', ''))
        
        # Clean conversation content as well
        cleaned_conversation = []
        for turn in conversation:
            cleaned_turn = {
                'role': turn.get('role', ''),
                'content': clean_text(turn.get('content', ''))
            }
            cleaned_conversation.append(cleaned_turn)
        
        csv_data.append({
            'dialog_id': dialog_id,
            'ground_truth': ground_truth,
            'conversation': cleaned_conversation
        })
    
    df = pd.DataFrame(csv_data)
    df.to_csv(output_file, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [8]:
current_directory = os.getcwd()
print(f"Current directory: {current_directory}")

# Load movie database mapping
movie_db_mapping = load_movie_database('raw/movie_database.tsv')

# Extract conversations from CSV and save directly to CSV
print("\nProcessing test data...")
test_conversations = extract_conversations('raw/test.tsv', movie_db_mapping)
save_conversations_to_csv(test_conversations, 'multiturn_form/test.csv')

print("\nProcessing train data...")
train_conversations = extract_conversations('raw/train.tsv', movie_db_mapping)
save_conversations_to_csv(train_conversations, 'multiturn_form/train.csv')

print(f"\nExtracted {len(train_conversations)} train conversations and {len(test_conversations)} test conversations")

# Print a sample conversation for verification
if train_conversations:
    print("\nSample conversation:")
    sample_conv = train_conversations[0]
    print(f"Dialog ID: {sample_conv['dialog_id']}")
    print(f"Ground Truth: {sample_conv['ground_truth']}")
    print("Conversation:")
    for i, turn in enumerate(sample_conv['conversation'][:4]):  # Show first 4 turns
        print(f"  {turn['role']}: {turn['content']}")

# Load and display dataset info
try:
    from datasets import Dataset, DatasetDict, load_dataset
    ds_dict = load_dataset("csv", data_files="multiturn_form/train.csv")
    print(f"\nDataset loaded with {len(ds_dict['train'])} examples")
    print("Sample ground truth values:")
    for i in range(min(5, len(ds_dict['train']))):
        print(f"  {ds_dict['train'][i]['ground_truth']}")
except ImportError:
    print("\nDatasets library not available for final verification")

Current directory: /home/sagemaker-user/csbai/multiturn_rl/datasets/inspired
Successfully loaded movie database with utf-8 encoding
Mapped 0rM5WnicOAE -> Antlers
Mapped KB_uCjht0nA -> Bloodshot
Mapped FF932ZU6Kn4 -> Jungle Cruise
Mapped Alv1znZA6Es -> Onward
Mapped VqCpR19iBpc -> The Turning
Mapped Ify9S7hj480 -> The Gentlemen
Mapped 5aq0o2C6LVE -> The King's Man
Mapped txDYwh46tew -> The Secret Garden
Mapped xOQioNJZ_qc -> Like a Boss
Mapped F9x4cb7P-hI -> Bad Boys for Life
Mapped Rszr56AH3Co -> Underwater
Mapped VbfC4lrNulE -> Top Gun: Maverick
Mapped mVgdfP7qj7s -> Mulan
Mapped JPzmECxyp_8 -> Trolls World Tour
Mapped HwT879-4Wjs -> The New Mutants
Mapped sE6fTeuKZkg -> Knives Out
Mapped 70UY9X7qhE0 -> 21 Bridges
Mapped Klw-rra8hOA -> The King
Mapped zvcKNaWgpXA -> Star Wars: The Rise of Skywalker
Mapped 5XTlfPKBIOM -> Marriage Story
Mapped cjXE_9ZmOpk -> The Report
Mapped B2G9-KhBek8 -> Bombshell
Mapped _EpH1zU-NIU -> Frozen II
Mapped mMcjf_1jhMc -> Earthquake Bird
Mapped nI9p58_KJm

Generating train split: 0 examples [00:00, ? examples/s]


Dataset loaded with 801 examples
Sample ground truth values:
  Knives Out
  Terminator: Dark Fate
  A Christmas Story 2
  Ready or Not
  The Grinch
